In [ ]:
from dotenv import load_dotenv
load_dotenv()

from anthropic import Anthropic
from typing import cast, Iterable
from anthropic.types import MessageParam
from typing import cast, Iterable


# Lessons 3: Making a Request

In [ ]:
# Making Your First Request

client = Anthropic()
model = "claude-sonnet-5-5"

message = client.messages.create(
    model = model,
    max_tokens = 1000,
    messages = cast(Iterable[MessageParam], cast(object, [
        {
            "role": "user",
            "content": "What is quantum computing? Answer in one sentence"
        }
    ]))
)

In [ ]:
message.content[0].text

# Lesson 4: Multi-Turn conversations

In [ ]:
# Building Helper Functions

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model = model,
        max_tokens = 1000,
        messages = messages,
    )
    return message.content[0].text

In [ ]:
# Putting it all together
messages = []

# First Turn
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
add_assistant(messages, answer)

# Second Turn
add_user_message(messages, "Write another sentence")
final_answer = chat(messages)
add_assistant(messages, final_answer)

In [ ]:
for msg in messages:
    print(f"{msg['role'].capitalize()}: {msg['content']}")

# Lesson 5: System Prompts

In [ ]:
messages = [
    {
        "role": "user",
        "content": "How do I solve 5x + 2 = 3 for x?"
    }
]

system_prompt = """
You are a patient math tutor.
Do not directly answer a student's questions.
Guide them to a solution step by step.
"""

# Building a Flexible Chat Function

def chat(messages, system = None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)

    if not message.content:
         return f"Error: No content returned. Stop reason: {message.stop_reason}"

    return message.content[0].text

In [ ]:
print("--- TESTING WITHOUT SYSTEM PROMPT ---")
answer_without = chat(messages)
print(answer_without)

In [ ]:
print("\n--- TESTING WITH SYSTEM PROMPT ---")
answer_with = chat(messages, system=system_prompt)
print(answer_with)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)
k = os.environ.get("ANTHROPIC_API_KEY", "")
print(k[:12], "...", k[-4:], "length:", len(k))